# Human Activity Recognition — how much of the accuracy is real?
**CS6140 term project (Track 3) — starter notebook**

Dataset: UCI HAR (Anguita et al., 2013). 30 subjects, smartphone on the waist, 6 activities,
2.56 s windows (128 samples at 50 Hz) with 50% overlap, plus 561 precomputed features per window.

Experiments in this notebook:
1. **E1 Baseline** — classical models on the official subject-wise split, compared to the published SVM result.
2. **E2 Leakage** — random window split vs. subject-grouped split on the same data.
3. **E3 Deep model** — 1D-CNN on raw inertial signals vs. handcrafted features.
4. **E4 Error analysis** — confusion matrix, per-subject accuracy (leave-one-subject-out).
5. **E5 Scaling** — accuracy vs. number of training subjects (the "what happens with more data" question).

Run top to bottom on Colab (Runtime → Change runtime type → GPU helps for E3 only).

In [ ]:
import os, zipfile, urllib.request, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 0
np.random.seed(SEED)

DATA_DIR = os.environ.get("HAR_DATA_DIR", "data")
ROOT = os.path.join(DATA_DIR, "UCI HAR Dataset")
OUT = "results"
os.makedirs(OUT, exist_ok=True)

CNN_EPOCHS = int(os.environ.get("CNN_EPOCHS", 30))
RUN_CNN = os.environ.get("RUN_CNN", "1") == "1"

## 1. Download and load the data

In [ ]:
URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]

def _extract_all(zpath, dest):
    # The UCI download is a zip that contains another zip; unpack recursively.
    with zipfile.ZipFile(zpath) as z:
        z.extractall(dest)
        inner = [n for n in z.namelist() if n.endswith(".zip") and not n.startswith("__MACOSX")]
    for n in inner:
        _extract_all(os.path.join(dest, n), dest)

def download():
    if os.path.isdir(ROOT):
        print("Data already present at", ROOT)
        return
    os.makedirs(DATA_DIR, exist_ok=True)
    zpath = os.path.join(DATA_DIR, "har.zip")
    for url in URLS:
        try:
            print("Downloading", url)
            urllib.request.urlretrieve(url, zpath)
            break
        except Exception as e:
            print("  failed:", e)
    else:
        raise RuntimeError("Download failed. Download the zip manually from the UCI repository "
                           "and unzip it so that '" + ROOT + "' exists.")
    _extract_all(zpath, DATA_DIR)
    assert os.path.isdir(ROOT), "Unzipped, but '" + ROOT + "' not found; check folder names."

download()

In [ ]:
SIGNALS = [f"{kind}_{axis}" for kind in ("body_acc", "body_gyro", "total_acc") for axis in "xyz"]
ACTIVITIES = ["WALKING", "UPSTAIRS", "DOWNSTAIRS", "SITTING", "STANDING", "LAYING"]

def _read(path, dtype=float):
    return pd.read_csv(path, sep=r"\s+", header=None).to_numpy(dtype=dtype)

def load_split(split):
    d = os.path.join(ROOT, split)
    X = _read(os.path.join(d, f"X_{split}.txt"))                        # (n, 561) handcrafted features
    y = _read(os.path.join(d, f"y_{split}.txt"), int).ravel() - 1       # labels 0..5
    s = _read(os.path.join(d, f"subject_{split}.txt"), int).ravel()     # subject ids 1..30
    raw = np.stack([_read(os.path.join(d, "Inertial Signals", f"{name}_{split}.txt"))
                    for name in SIGNALS], axis=1).astype(np.float32)    # (n, 9, 128) raw signals
    return X, y, s, raw

Xtr, ytr, str_, Rtr = load_split("train")
Xte, yte, ste, Rte = load_split("test")

# Pooled copy (all 30 subjects) for the cross-validation experiments
Xall = np.vstack([Xtr, Xte]); yall = np.concatenate([ytr, yte])
sall = np.concatenate([str_, ste]); Rall = np.concatenate([Rtr, Rte])

### Data summary (this is the evidence for the proposal that the data is in hand)

In [ ]:
print("Train windows:", Xtr.shape, "| raw:", Rtr.shape, "| subjects:", sorted(map(int, set(str_))))
print("Test  windows:", Xte.shape, "| raw:", Rte.shape, "| subjects:", sorted(map(int, set(ste))))
print("Subject overlap between official train and test:", set(str_) & set(ste) or "none")
print(pd.Series(yall).map(dict(enumerate(ACTIVITIES))).value_counts().sort_index())

## 2. E1 — Baseline on the official subject-wise split
External ground truth: Anguita et al. (2013) report roughly 96% test accuracy with a multiclass SVM on these
561 features. Check the exact number in the paper and cite it in the report.

In [ ]:
def make_models():
    return {
        "LogReg":   make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
        "SVM-RBF":  make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale")),
        "RandomForest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
        "HistGB":   HistGradientBoostingClassifier(random_state=SEED),
    }

def score(y_true, y_pred):
    return {"acc": accuracy_score(y_true, y_pred), "macro_f1": f1_score(y_true, y_pred, average="macro")}

In [ ]:
e1 = {}
fitted = {}
for name, m in make_models().items():
    t = time.time()
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    e1[name] = {**score(yte, pred), "fit_sec": round(time.time() - t, 1)}
    fitted[name] = m
    print(f"{name:13s} acc={e1[name]['acc']:.4f}  macroF1={e1[name]['macro_f1']:.4f}  ({e1[name]['fit_sec']}s)")

e1_df = pd.DataFrame(e1).T
e1_df.to_csv(f"{OUT}/e1_official_split.csv")
e1_df

## 3. E2 — Does a random split inflate accuracy?
Windows overlap by 50%, so neighbouring windows share half their samples, and they come from the same person.
A shuffled split puts near-duplicates of test windows in the training set. A subject-grouped split does not.
Same data, same models, only the split changes.

In [ ]:
def cv_compare(models, X, y, groups, n_splits=5):
    rows = []
    splitters = {
        "random (shuffled windows)": StratifiedKFold(n_splits, shuffle=True, random_state=SEED).split(X, y),
        "grouped (by subject)":      GroupKFold(n_splits).split(X, y, groups),
    }
    for split_name, folds in splitters.items():
        folds = list(folds)
        for name, m in models.items():
            accs = []
            for tr, te in folds:
                m.fit(X[tr], y[tr]); accs.append(accuracy_score(y[te], m.predict(X[te])))
            rows.append({"split": split_name, "model": name, "acc_mean": np.mean(accs), "acc_std": np.std(accs)})
            print(f"{split_name:27s} {name:13s} {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    return pd.DataFrame(rows)

e2_models = {k: v for k, v in make_models().items() if k != "SVM-RBF"}   # SVM is slow in CV; add back if time allows
e2_df = cv_compare(e2_models, Xall, yall, sall)
e2_df.to_csv(f"{OUT}/e2_split_leakage.csv", index=False)

pivot = e2_df.pivot(index="model", columns="split", values="acc_mean")
pivot["gap (random - grouped)"] = pivot["random (shuffled windows)"] - pivot["grouped (by subject)"]
pivot

## 4. E3 — 1D-CNN on raw signals
Learns features from the 9×128 raw signals instead of using the 561 handcrafted ones.
Normalization statistics come from the training subjects only.

In [ ]:
if RUN_CNN:
    import torch, torch.nn as nn
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)

    class HARCNN(nn.Module):
        def __init__(self, n_ch=9, n_cls=6):
            super().__init__()
            def block(i, o, k): return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2), nn.BatchNorm1d(o), nn.ReLU())
            self.net = nn.Sequential(block(n_ch, 64, 5), block(64, 64, 5), nn.MaxPool1d(2),
                                     block(64, 128, 3), nn.AdaptiveAvgPool1d(1), nn.Flatten(),
                                     nn.Dropout(0.3), nn.Linear(128, n_cls))
        def forward(self, x): return self.net(x)

    def train_cnn(Rtrain, ytrain, Rtest, ytest, epochs=CNN_EPOCHS, bs=64, lr=1e-3):
        mu = Rtrain.mean(axis=(0, 2), keepdims=True); sd = Rtrain.std(axis=(0, 2), keepdims=True) + 1e-8
        to_t = lambda a: torch.tensor((a - mu) / sd, dtype=torch.float32)
        Xa, Xb = to_t(Rtrain), to_t(Rtest)
        ya = torch.tensor(ytrain, dtype=torch.long)
        model = HARCNN().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        lossf = nn.CrossEntropyLoss()
        history = []
        for ep in range(epochs):
            model.train()
            perm = torch.randperm(len(Xa))
            for i in range(0, len(Xa), bs):
                idx = perm[i:i + bs]
                opt.zero_grad()
                loss = lossf(model(Xa[idx].to(device)), ya[idx].to(device))
                loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                pred = torch.cat([model(Xb[i:i + 512].to(device)).argmax(1).cpu() for i in range(0, len(Xb), 512)]).numpy()
            history.append(accuracy_score(ytest, pred))
            if (ep + 1) % 5 == 0 or ep == epochs - 1:
                print(f"epoch {ep + 1:3d}  test acc {history[-1]:.4f}")
        return pred, history

    cnn_pred, cnn_hist = train_cnn(Rtr, ytr, Rte, yte)
    e1_df.loc["1D-CNN (raw)"] = {**score(yte, cnn_pred), "fit_sec": np.nan}
    e1_df.to_csv(f"{OUT}/e1_official_split.csv")
    # Note: watching test accuracy every epoch is for monitoring only. For the report, pick the epoch count
    # with a validation split of held-out *training subjects*, not the test set.
e1_df

## 5. E4 — Error analysis

In [ ]:
best = e1_df["acc"].drop(labels=["1D-CNN (raw)"], errors="ignore").idxmax()
pred = fitted[best].predict(Xte)
cm = confusion_matrix(yte, pred, normalize="true")

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(6), ACTIVITIES, rotation=45, ha="right"); ax.set_yticks(range(6), ACTIVITIES)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", color="white" if cm[i, j] > 0.5 else "black", fontsize=8)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"Confusion matrix — {best} (official test)")
plt.colorbar(im); plt.tight_layout(); plt.savefig(f"{OUT}/e4_confusion.png", dpi=150); plt.show()

print(classification_report(yte, pred, target_names=ACTIVITIES, digits=3))

In [ ]:
# Leave-one-subject-out: how much does accuracy vary from person to person?
loso_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
per_subject = {}
for tr, te in LeaveOneGroupOut().split(Xall, yall, sall):
    loso_model.fit(Xall[tr], yall[tr])
    per_subject[int(sall[te][0])] = accuracy_score(yall[te], loso_model.predict(Xall[te]))
ps = pd.Series(per_subject).sort_values()
ps.to_csv(f"{OUT}/e4_loso_per_subject.csv")
print(f"LOSO mean {ps.mean():.4f}, worst subject {ps.idxmin()} ({ps.min():.4f}), best {ps.idxmax()} ({ps.max():.4f})")

plt.figure(figsize=(9, 3))
plt.bar(ps.index.astype(str), ps.values); plt.axhline(ps.mean(), ls="--", c="k", lw=1)
plt.ylim(max(0, ps.min() - 0.05), 1.0); plt.xlabel("subject (sorted)"); plt.ylabel("accuracy")
plt.title("Leave-one-subject-out accuracy (LogReg)"); plt.tight_layout(); plt.savefig(f"{OUT}/e4_loso.png", dpi=150); plt.show()

## 6. E5 — Accuracy vs. number of training subjects
Train on k randomly chosen training subjects, always test on the 9 official test subjects. Repeat with several
random subject draws so the error bars reflect *which* people you happened to collect.

In [ ]:
train_subjects = np.array(sorted(map(int, set(str_))))
ks = [k for k in (2, 4, 8, 12, 16, len(train_subjects)) if k <= len(train_subjects)]
rng = np.random.default_rng(SEED)
rows = []
for k in ks:
    for rep in range(5 if k < len(train_subjects) else 1):
        chosen = rng.choice(train_subjects, size=k, replace=False)
        mask = np.isin(str_, chosen)
        for name in ("LogReg", "RandomForest"):
            m = make_models()[name]
            m.fit(Xtr[mask], ytr[mask])
            rows.append({"k": k, "rep": rep, "model": name, "acc": accuracy_score(yte, m.predict(Xte))})
e5_df = pd.DataFrame(rows)
e5_df.to_csv(f"{OUT}/e5_learning_curve.csv", index=False)

agg = e5_df.groupby(["model", "k"])["acc"].agg(["mean", "std"]).reset_index()
plt.figure(figsize=(6, 4))
for name, g in agg.groupby("model"):
    plt.errorbar(g["k"], g["mean"], yerr=g["std"].fillna(0), marker="o", capsize=3, label=name)
plt.xlabel("number of training subjects"); plt.ylabel("test accuracy (unseen subjects)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.savefig(f"{OUT}/e5_learning_curve.png", dpi=150); plt.show()
agg

## Next steps (fill in as you go)
- Tune hyperparameters with **GroupKFold on training subjects only**; never touch the official test set while tuning.
- Run the CNN under both splits in E2 as well (does a deep model exploit the leak more or less than RF?).
- Optional stretch: test on a second dataset (e.g. WISDM or PAMAP2) for cross-dataset generalization.